# 2026a2_songs.json Analysis

Raw data exploration for the songs dataset.

In [1]:
import json
import pandas as pd

In [2]:
with open('2026a2_songs.json', 'r', encoding='utf-8') as f:
    songs_data = json.load(f)

df = pd.json_normalize(songs_data['songs'])
df.head()

,title,artist,year,album,img_url
0,1904,The Tallest Man on Earth,2012,There's No Leaving Now,https://raw.githubusercontent.com/YingZhang201...
1,#40,Dave Matthews,1999,Listener Supported,https://raw.githubusercontent.com/YingZhang201...
2,40oz to Freedom,Sublime,1996,40oz. to Freedom,https://raw.githubusercontent.com/YingZhang201...
3,#41,Dave Matthews,1996,Crash,https://raw.githubusercontent.com/YingZhang201...
4,American Girl,Tom Petty,1977,Tom Petty and the Heartbreakers,https://raw.githubusercontent.com/YingZhang201...


In [3]:
df.shape

(137, 5)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    137 non-null    str  
 1   artist   137 non-null    str  
 2   year     137 non-null    str  
 3   album    137 non-null    str  
 4   img_url  137 non-null    str  
dtypes: str(5)
memory usage: 5.5 KB


In [5]:
print(f'Total rows: {len(df)}')
print(f'Column names: {df.columns.tolist()}')

Total rows: 137
Column names: ['title', 'artist', 'year', 'album', 'img_url']


The following attributes with 137 records stored inside `2026a2_songs.json`:
- **title**: Title of the song
- **artist**: Artist name
- **year**: Year of when was the song is released
- **album**: The name of the album where the single(songs) is part of
- **img_url**: Cover art

In [6]:
# Count unique values from attributes
print(f'Unique titles: {df['title'].nunique()}')
print(f'Unique artists: {df['artist'].nunique()}')
print(f'Unique albums: {df['album'].nunique()}')

Unique titles: 130
Unique artists: 71
Unique albums: 102


By comparing the values above with the total records in the dataset, it indicates that the song titles, artits and albums are not unique. Which could indicate:
1. Different artists could have the same song and album titles
2. Multiple songs are part of the same album
3. Multiple songs/albums could be coming from the same artist(s)

In [7]:
# Song titles that appeared multiple times
df[df.duplicated(subset=['title'], keep=False)].sort_values('title')[['title', 'artist', 'album']]

,title,artist,album
10,Bad Blood,Taylor Swift,1989
11,Bad Blood,Kendrick Lamar,To Pimp a Butterfly
26,Delicate,Taylor Swift,Reputation
27,Delicate,Taylor Swift,Reputation (Deluxe)
65,I Won't Give Up,Jason Mraz,Love is a Four Letter Word
66,I Won't Give Up,Jason Mraz,Lalalalovesongs (Re:2021)
93,Rivers of Babylon,Sublime,Acoustic: Bradley Nowell & Friends
94,Rivers of Babylon,The Melodians,Rivers of Babylon
95,Rivers of Babylon,The Melodians,Rivers of Babylon (Re-release)
117,The Needle and the Damage Done,Neil Young,Harvest


In [8]:
# Show artist re-using the same song titles
df[df.duplicated(subset=['artist', 'title'], keep=False)]

,title,artist,year,album,img_url
26,Delicate,Taylor Swift,2017,Reputation,https://raw.githubusercontent.com/YingZhang201...
27,Delicate,Taylor Swift,2018,Reputation (Deluxe),https://raw.githubusercontent.com/YingZhang201...
65,I Won't Give Up,Jason Mraz,2012,Love is a Four Letter Word,https://raw.githubusercontent.com/YingZhang201...
66,I Won't Give Up,Jason Mraz,2021,Lalalalovesongs (Re:2021),https://raw.githubusercontent.com/YingZhang201...
94,Rivers of Babylon,The Melodians,1970,Rivers of Babylon,https://raw.githubusercontent.com/YingZhang201...
95,Rivers of Babylon,The Melodians,2003,Rivers of Babylon (Re-release),https://raw.githubusercontent.com/YingZhang201...
128,We Are Never Ever Getting Back Together,Taylor Swift,2012,Red,https://raw.githubusercontent.com/YingZhang201...
129,We Are Never Ever Getting Back Together,Taylor Swift,2013,Red (Deluxe Edition),https://raw.githubusercontent.com/YingZhang201...


In [9]:
# Justifying the schema
def check_unique(cols):
    return df[cols].drop_duplicates().shape[0]

total = len(df)

print(f'title: {check_unique(["title"]) == total}')
print(f'artist: {check_unique(["artist"]) == total}')
print(f'album: {check_unique(["album"]) == total}')
print(f'title + artist: {check_unique(["title", "artist"]) == total}')
print(f'artist + album: {check_unique(["artist", "album"]) == total}')
"""
The code below should print out True because the schema key from create_music_table.py is
title_album = title + # + album => (artist, title#album)
which is the same as: (artist, title, album)
"""
print(f'artist + title + album: {check_unique(["artist", "title", "album"]) == total}')

title: False
artist: False
album: False
title + artist: False
artist + album: False
artist + title + album: True


In [10]:
df['title_album'] = df['title'] + '#' + df['album']
(df[['artist', 'title_album']].drop_duplicates().shape[0] == len(df))

True

The cell block above suggests that the schema is loseless.

Primary Key (PK) = `artist` and Surrogate Key (SK) = `title` is not safe because the combination `(artist, title)` is not unique in the dataset. Some artists have the same song title appearing more than once across different albums or releases. In DynamoDB, items with the same partition key and sort key are treated as the same item, so one record could overwrite another during insertion. Therefore, using only artist and title would not guarantee a lossless import.